In [12]:
# tools/reconstruct_port_relations.py
# -*- coding: utf-8 -*-
"""
IFC2x3 のモデルに対して, 与えられた GUID の有向リストから
IfcRelConnectsPortToElement と IfcRelConnectsPorts を擬似的に再現する
再生成前に, 経路に関与する既存ポート関連記述を削除するクリーンアップ機能を提供する
"""

from pathlib import Path
from typing import Optional, Iterable, List
import re

import ifcopenshell
from ifcopenshell.guid import new as new_guid

# ========== 設定 ==========
IFC_IN = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project\data\ifc\raw\51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01.ifc")
IFC_OUT = IFC_IN.with_name(IFC_IN.stem + "_with_port_relations.ifc")

# 例: AHU → ... → FlowTerminal の GUID パス
PATH_GUIDS: Iterable[str] = [
    "2xh2uwStv1sh1c4KjqB4JW",
    "00SWBOwCbDSuN3bGIZHDol",
    "00SWBOwCbDSuN3bGIZHDos",
    "00SWBOwCbDSuN3bGIZHDwO",
    "3T5NVXezz9DRBS$A4FkwyQ",
    "00SWBOwCbDSuN3bGIZHDxI",
    "3T5NVXezz9DRBS$A4FkwyO",
    "3T5NVXezz9DRBS$A4FkwyR",
    "3T5NVXezz9DRBS$A4Fkw52",
    "3T5NVXezz9DRBS$A4FkwR0",
    "3T5NVXezz9DRBS$A4FkwTZ",
    "3T5NVXezz9DRBS$A4FkwTW",
    "2JOW00849CcP9in0o$ujFe",
    "2JOW00849CcP9in0o$ujC8",
    "2JOW00849CcP9in0o$ujDe",
    "2JOW00849CcP9in0o$ujDh",
    "2JOW00849CcP9in0o$ujFn",
    "2JOW00849CcP9in0o$ujCF",
    "1uFV_RvcX8$uvgkDx1PuVu",
    "3T5NVXezz9DRBS$A4FkwS0",
    "3T5NVXezz9DRBS$A4FkwFH",
    "3T5NVXezz9DRBS$A4Fkw6L",
    "3T5NVXezz9DRBS$A4FkwFN",
    "3T5NVXezz9DRBS$A4FkwFK",
    "3T5NVXezz9DRBS$A4Fkw6N",
    "3T5NVXezz9DRBS$A4Fkw6R",
    "3T5NVXezz9DRBS$A4Fkw6P",
    "3T5NVXezz9DRBS$A4FkwOi",
    "3T5NVXezz9DRBS$A4FkwOl",
    "3T5NVXezz9DRBS$A4FkwOk",
    "1uFV_RvcX8$uvgkDx1PuUp",
    "3T5NVXezz9DRBS$A4Fkw67",
    "3T5NVXezz9DRBS$A4Fkw65",
    "3T5NVXezz9DRBS$A4Fkw6B",
    "3T5NVXezz9DRBS$A4Fkw69",
    "3T5NVXezz9DRBS$A4Fkw6D",
    "3T5NVXezz9DRBS$A4FkwNc",
]

# ========== ユーティリティ ==========

def get_owner_history(model) -> Optional[object]:
    oh = model.by_type("IfcOwnerHistory")
    return oh[0] if oh else None

def ensure_owner_history(model):
    oh = get_owner_history(model)
    if oh:
        return oh
    person = model.create_entity("IfcPerson", Identification="M51", FamilyName="Morgate51")
    org = model.create_entity("IfcOrganization", Identification="M51ORG", Name="Morgate51 Project")
    actor = model.create_entity("IfcPersonAndOrganization", ThePerson=person, TheOrganization=org)
    app = model.create_entity("IfcApplication", ApplicationDeveloper=org, Version="0.1",
                              ApplicationFullName="M51 PortLinker", ApplicationIdentifier="M51PL")
    oh = model.create_entity("IfcOwnerHistory", OwningUser=actor, OwningApplication=app, ChangeAction="ADDED")
    return oh

def get_element_by_guid(model, guid: str):
    try:
        e = model.by_guid(guid)
        if e:
            return e
    except Exception:
        pass
    for e in model.by_type("IfcRoot"):
        if getattr(e, "GlobalId", None) == guid:
            return e
    return None

def iter_element_ports(model, element) -> Iterable[object]:
    """
    要素に取り付く IfcDistributionPort を列挙する
    IfcRelConnectsPortToElement の RelatingPort をたどる
    """
    if not hasattr(element, "HasPorts") or not element.HasPorts:
        return []
    for rel in element.HasPorts:
        if rel.is_a("IfcRelConnectsPortToElement"):
            port = rel.RelatingPort if hasattr(rel, "RelatingPort") else None
            if port:
                yield port

def iter_port_links(model, port) -> Iterable[object]:
    """
    ポートに接続する IfcRelConnectsPorts を列挙する
    """
    for rel in model.by_type("IfcRelConnectsPorts"):
        if rel.RelatingPort == port or rel.RelatedPort == port:
            yield rel

def cleanup_existing_port_graph(model, path_guids: Iterable[str], mode: str = "aggressive") -> None:
    """
    経路 PATH_GUIDS に含まれる要素に取り付く既存ポートおよびポート間接続を削除する
    mode='aggressive' は当該要素の全ポートを削除
    mode='safe' は Name が ^(IN|OUT)_\\d+$ のポートのみ削除
    """
    name_pat = re.compile(r"^(IN|OUT)_\\d+$")
    targets_ports: List[object] = []
    targets_rel_pe: List[object] = []   # IfcRelConnectsPortToElement
    targets_rel_pp: List[object] = []   # IfcRelConnectsPorts

    guid_set = set(path_guids)
    for guid in guid_set:
        el = get_element_by_guid(model, guid)
        if el is None:
            print(f"警告, PATH 内 GUID が IFC に見つからない, guid={guid}, 要検討")
            continue
        if not hasattr(el, "HasPorts") or not el.HasPorts:
            continue
        for rel_pe in el.HasPorts:
            if not rel_pe.is_a("IfcRelConnectsPortToElement"):
                continue
            port = rel_pe.RelatingPort if hasattr(rel_pe, "RelatingPort") else None
            if port is None:
                continue
            # safe モードでは IN_*, OUT_* 名称のみ対象
            if mode == "safe":
                if not getattr(port, "Name", None) or not name_pat.match(str(port.Name)):
                    continue
            targets_ports.append(port)
            targets_rel_pe.append(rel_pe)
            for rel_pp in iter_port_links(model, port):
                targets_rel_pp.append(rel_pp)

    # 重複排除
    targets_ports = list(dict.fromkeys(targets_ports))
    targets_rel_pe = list(dict.fromkeys(targets_rel_pe))
    targets_rel_pp = list(dict.fromkeys(targets_rel_pp))

    # 依存関係順で削除, 先にポート間接続, 次にエレメント接続, 最後にポート本体
    for rel in targets_rel_pp:
        model.remove(rel)
    for rel in targets_rel_pe:
        model.remove(rel)
    for p in targets_ports:
        model.remove(p)

    print(f"cleanup, removed IfcRelConnectsPorts={len(targets_rel_pp)}, "
          f"IfcRelConnectsPortToElement={len(targets_rel_pe)}, Ports={len(targets_ports)}")

def has_existing_port_connection(model, up_port, dn_port) -> bool:
    for rel in model.by_type("IfcRelConnectsPorts"):
        if rel.RelatingPort == up_port and rel.RelatedPort == dn_port:
            return True
    return False

def get_or_create_port(model, element, flow_dir: str, name: str, owner_history) -> object:
    if hasattr(element, "HasPorts") and element.HasPorts:
        for rel in element.HasPorts:
            if rel.is_a("IfcRelConnectsPortToElement"):
                p = rel.RelatingPort if hasattr(rel, "RelatingPort") else None
                if p and getattr(p, "FlowDirection", None) == flow_dir and getattr(p, "Name", None) == name:
                    return p
    port = model.create_entity(
        "IfcDistributionPort",
        GlobalId=new_guid(),
        OwnerHistory=owner_history,
        Name=name,
        FlowDirection=flow_dir,
    )
    model.create_entity(
        "IfcRelConnectsPortToElement",
        GlobalId=new_guid(),
        OwnerHistory=owner_history,
        Name=f"{element.GlobalId}-{name}",
        RelatingPort=port,
        RelatedElement=element,
    )
    return port

def connect_ports(model, up_port, down_port, name: str, owner_history):
    if has_existing_port_connection(model, up_port, down_port):
        return
    model.create_entity(
        "IfcRelConnectsPorts",
        GlobalId=new_guid(),
        OwnerHistory=owner_history,
        Name=name,
        RelatingPort=up_port,
        RelatedPort=down_port,
    )

def reconstruct_chain(model, path_guids):
    oh = ensure_owner_history(model)
    if oh is None:
        raise RuntimeError("OwnerHistory の確保に失敗した, 要検討")

    for i in range(len(path_guids) - 1):
        up_guid = path_guids[i]
        dn_guid = path_guids[i + 1]
        up_el = get_element_by_guid(model, up_guid)
        dn_el = get_element_by_guid(model, dn_guid)
        if up_el is None or dn_el is None:
            print(f"警告, 要素未検出, up={up_guid}, dn={dn_guid}, 要検討")
            continue

        up_port = get_or_create_port(model, up_el, "SOURCE", f"OUT_{i+1}", owner_history=oh)
        dn_port = get_or_create_port(model, dn_el, "SINK",   f"IN_{i+1}",  owner_history=oh)

        connect_ports(model, up_port, dn_port, name=f"{up_el.GlobalId}->{dn_el.GlobalId}", owner_history=oh)
        print(f"connected: {up_el.GlobalId} -> {dn_el.GlobalId}")

def main():
    model = ifcopenshell.open(str(IFC_IN))
    print("Loaded:", IFC_IN.name, "Schema:", model.schema)
    # 1) 経路に関与する既存のポート関連記述を削除, safe または aggressive を選択
    cleanup_existing_port_graph(model, PATH_GUIDS, mode="aggressive")
    # 2) ポート関係を再構築
    reconstruct_chain(model, list(PATH_GUIDS))
    # 3) 保存
    model.write(str(IFC_OUT))
    print("Saved:", IFC_OUT)

if __name__ == "__main__":
    main()

Loaded: 51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01.ifc Schema: IFC2X3
cleanup, removed IfcRelConnectsPorts=35, IfcRelConnectsPortToElement=84, Ports=84
connected: 2xh2uwStv1sh1c4KjqB4JW -> 00SWBOwCbDSuN3bGIZHDol
connected: 00SWBOwCbDSuN3bGIZHDol -> 00SWBOwCbDSuN3bGIZHDos
connected: 00SWBOwCbDSuN3bGIZHDos -> 00SWBOwCbDSuN3bGIZHDwO
connected: 00SWBOwCbDSuN3bGIZHDwO -> 3T5NVXezz9DRBS$A4FkwyQ
connected: 3T5NVXezz9DRBS$A4FkwyQ -> 00SWBOwCbDSuN3bGIZHDxI
connected: 00SWBOwCbDSuN3bGIZHDxI -> 3T5NVXezz9DRBS$A4FkwyO
connected: 3T5NVXezz9DRBS$A4FkwyO -> 3T5NVXezz9DRBS$A4FkwyR
connected: 3T5NVXezz9DRBS$A4FkwyR -> 3T5NVXezz9DRBS$A4Fkw52
connected: 3T5NVXezz9DRBS$A4Fkw52 -> 3T5NVXezz9DRBS$A4FkwR0
connected: 3T5NVXezz9DRBS$A4FkwR0 -> 3T5NVXezz9DRBS$A4FkwTZ
connected: 3T5NVXezz9DRBS$A4FkwTZ -> 3T5NVXezz9DRBS$A4FkwTW
connected: 3T5NVXezz9DRBS$A4FkwTW -> 2JOW00849CcP9in0o$ujFe
connected: 2JOW00849CcP9in0o$ujFe -> 2JOW00849CcP9in0o$ujC8
connected: 2JOW00849CcP9in0o$ujC8 -> 2JOW00849CcP9in0o$ujDe
connected: 2J